## 导读

本notebook主要介绍了一种分析复杂网络数据的半监督算法——图卷积神经网络（GCN）：该算法可以用于对复杂网络数据进行建模分析，且建模过程只需要部分样本有标签即可（半监督），这很符合金融风控中来自团伙作案的风险场景。本notebook以一个简单案例的形式对该算法原理进行了介绍，最后分析了该算法在金融风控中的应用价值。

### **背景**

风控一直以来都是金融行业的重中之重，金融风险按照参与人数可以分为来自个人的风险和来自团伙的风险：来自个人的风险主是指个人造成的风险，如个人信用卡违约；来自团伙的风险主要是团伙作案造成的风险，如团伙欺诈。为了更好控制风险，金融企业和相关研究人员将机器学习算法引入金融行业的风控领域，这极大提高了风控水平。但是金融风控领域的数据有一个很大的特点就是正负比例通常极不均衡，因为有风险的客户数量相对全体客户来说占比非常小，这种极不均衡的数据并不适合用传统的机器学习模型来处理，无监督或半监督模型可能会更适合。

建模的原则是根据不同的数据建立相对应的模型：对于个体风险，可以使用检测个体异常的无监督算法，如孤立森林；对于团伙风险，由于这类场景中样本之间存在关联，且样本间的关联包含了重要的风险信息（如团伙欺诈），但是个体的异常检测算法难以捕捉样本间的关联信息，此时可以将样本之间的关联信息组成复杂网络的形式，再使用针对复杂网络算法来处理这类数据，比如我们今天将要跟大家介绍的图卷积神经网络。

### **理论介绍**

**为什么有图卷积神经网络**

自2012年以来，深度学习在计算机视觉以及自然语言处理两个领域取得了巨大的成功。和传统方法相比，它好在哪里呢？



假设有一张图，要做分类，传统方法需要手动提取一些特征，比如纹理啊，颜色啊，或者一些更高级的特征。然后再把这些特征放到像随机森林等分类器，给到一个输出标签，告诉它是哪个类别。而深度学习是输入一张图，经过神经网络，直接输出一个标签。特征提取和分类一步到位，避免了手工提取特征或者人工规则，从原始数据中自动化地去提取特征，是一种端到端（end-to-end）的学习。相较于传统的方法，深度学习能够学习到更高效的特征与模式。

![img](./img/gcn1.jpeg)



卷积神经网络很好，但是它研究的对象还是限制在Euclidean domains的数据。什么是Euclidean data？ Euclidean data最显著的特征就是有规则的空间结构，比如图片是规则的正方形栅格，比如语音是规则的一维序列。而这些数据结构能够用一维、二维的矩阵表示，卷积神经网络处理起来很高效。



![img](./img/gcn2.jpeg)



但是，我们的现实生活中有很多数据并不具备规则的空间结构，称为Non Euclidean data。比如推荐系统、电子交易、计算几何、脑信号、分子结构等抽象出的图谱。这些图谱结构每个节点连接都不尽相同，有的节点有三个连接，有的节点有两个连接，是不规则的数据结构。



下面结合两个典型的业务场景来说明什么是图：

![img](./img/gcn3.jpeg)

上面的图谱刻画社交网络中各个节点以及它们之间的关系，用户A、用户B、帖子都是节点，用户与用户之间的关系是关注，用户与帖子之间的关系可能是发布或者转发。通过这样一个图谱，可以分析用户对什么人、什么事感兴趣，进一步实现推荐机制。



![img](./img/gcn4.jpeg)

在电商中，我们首先可以想到的关键节点就是，用户、交易和商品。用户关联的节点比如会有注册地址、收获地址等；交易会关联到商品、收货地址、交易IP等、商品会关联类目等。这些节点之间的关系，比如用户除了可以通过交易购买商品，还可以对商品进行评分。这样的图数据我们可以用来做两件事情，一是推荐、二是反欺诈。



通过上面两个例子，可以很明显的感受到，图有两个基本的特性：

**一是每个节点都有自己的特征信息。**比如针对上图，我们建立一个风控规则，要看这个用户的注册地址、IP地址、交易的收货地址是否一样，如果这些特征信息不匹配，那么系统就会判定这个用户就存在一定的欺诈风险。这是对图节点特征信息的应用。



**二是图谱中的每个节点还具有结构信息。**如果某段时间某个IP节点连接的交易节点非常多，也就是说从某个IP节点延伸出来的边非常多，那么风控系统会判定这个IP地址存在风险。这是对图节点结构信息的应用。



总的来说，在图数据里面，我们要同时考虑到节点的特征信息以及结构信息，如果靠手工规则来提取，必将失去很多隐蔽和复杂的模式，那么有没有一种方法能自动化地同时学到图的特征信息与结构信息呢？——图卷积神经网络



**什么是图卷积神经网络**

图卷积神经网络（Graph Convolutional Network）是一种能对图数据进行深度学习的方法。



![img](./img/gcn5.jpeg)

**图卷积算子：**


<img src="./img/gcn6.jpeg" style="zoom:18%;" />



上面给出的是图卷积算子的计算公式，设中心节点为i；


<img src="./img/gcn7.jpeg" style="zoom:30%;" />



**如何理解图卷积算法？我们看动图分三步去理解（注意不同颜色代表不同的权重）：**

第一步：发射（send）每一个节点将自身的特征信息经过变换后发送给邻居节点。这一步是在对节点的特征信息进行抽取变换。



![img](./img/gcn8.gif)


第二步：接收（receive）每个节点将邻居节点的特征信息聚集起来。这一步是在对节点的局部结构信息进行融合。



![img](./img/gcn9.gif)



第三步：变换（transform）把前面的信息聚集之后做非线性变换，增加模型的表达能力。



![img](./img/gcn10.gif)





**图卷积神经网络具有卷积神经网络的以下性质：**

1、局部参数共享，算子是适用于每个节点（圆圈代表算子），处处共享。

2、感受域正比于层数，最开始的时候，每个节点包含了直接邻居的信息，再计算第二层时就能把邻居的邻居的信息包含进来，这样参与运算的信息就更多更充分。层数越多，感受域就更广，参与运算的信息就更多。

<img src="./img/gcn11.gif" style="zoom:50%;" />



我们来看GCN这个模型框架，输入是一张图，经过一层一层计算变换，最后输出一张图。

<img src="./img/gcn12.jpeg" style="zoom:50%;" />



**GCN模型同样具备深度学习的三种性质：**

1、层级结构（特征一层一层抽取，一层比一层更抽象，更高级）；

2、非线性变换 （增加模型的表达能力）；

3、端对端训练（不需要再去定义任何规则，只需要给图的节点一个标记，让模型自己学习，融合特征信息和结构信息。）



**GCN四个特征：**

1、GCN 是对卷积神经网络在 graph domain 上的自然推广。

2、它能同时对节点特征信息与结构信息进行端对端学习，是目前对图数据学习任务的最佳选择。

3、图卷积适用性极广，适用于任意拓扑结构的节点与图。

4、在节点分类与边预测等任务上，在公开数据集上效果要远远优于其他方法。



**我们怎么用图卷积神经网络**

下面分享一个我们在实际应用场景中的实验：


<img src="./img/gcn13.jpeg" style="zoom:40%;" />


实验输入是一个验证数据构成的图数据，节点是验证事件以及事件相关的属性节点。如IP，DeviceID，UA等节点。（我们总计用了30天的验证数据，每两个小时的数据构成一张图，共360张图。）



实验输出是对事件节点进行人机分类，正常或者异常。

**实验细节**

**网络结构：**
GCN(128)->GCN(64)->GCN(64)->Linear(2)

**训练:** Adam优化器, lr=0.001
**参照基准:** 以只能学习特征信息的GBDT做为基准, grid_search 搜索超参数，GBDT是目前最流行的浅层分类器。



我们用第一天的数据做训练，持续30天预测结果如下：

<img src="./img/gcn14.jpeg" style="zoom:40%;" />



GCN模型的准确率衰减比较小，而GBDT的衰减很严重。可见，GCN模型的人机判别效果要好，鲁棒性好。

<img src="./img/gcn15.jpeg" style="zoom:40%;" />


7d评估效果可视化，（用第一天的数据训练模型，第七天观察其预测效果及最后一层输出的tsne可视化结果)。上图可以看出，GCN在第七天时对样本判别的分界面仍很明显，但是GBDT对样本判别的分界面已经很模糊类了。综上，GCN学到的结构信息在人机判别中不仅效果很好，也具有更好的鲁棒性。

### **应用**

GCN在**金融风控领域**，尤其是**团伙欺诈**和**反洗钱**等场景有很好的应用：

- **半监督**：根据GCN论文中的实验结果和使用GCN进行实际应用的效果显示，GCN只需要少量有标签的样本就可以训练出精度高的模型。在金融风控领域，我们往往只知道少量样本的真实标签（如发生过违约的用户，可以确定有风险），目前没有违约的用户很难确定是否有风险。传统的机器学习分类算法不适合这种正负比例极不均衡数据，GCN这种半监督模型面对极不均衡数据时效果通常会更好。
- **复杂网络数据**：金融风控中有一部分风险是来自团伙作案，传统的模型大多只能分析单个样本的风险信息，而GCN能将样本之间的关联信息（如社交网络、共用电子设备、财务往来等）利用起来，适用于金融风控中团伙作造成风险的场景。
- **案例**：根据企业自身的财务状况信息和企业之间的金融交易数据构建一个复杂网络，再使用GCN来处理这类数据，从而识别哪些交易具有欺诈性。例如，可以建立一个交易的超级节点图，判断是否有洗钱圈？找出洗钱团伙：

<img src="./img/gcn16.jpeg" style="zoom:60%;" />